# 062 — Detección, segmentación y pose

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Tareas:** detección (caja + clase + confianza), segmentación semántica (clase por píxel,
sin instancias), segmentación de instancias (máscara por objeto), pose (K keypoints).

**IoU** mide solapamiento: `IoU = área(A∩B) / área(A∪B)`; una detección es TP si
`IoU > 0.5` (típico). **NMS** depura duplicados: acepta la caja de mayor confianza y
elimina las que se solapan demasiado con ella. **mAP** ordena detecciones por confianza,
traza precisión-recall por clase y promedia el área (COCO promedia además IoU 0.5–0.95).

**Arquitecturas:** dos etapas (Faster R-CNN: red de propuestas + clasificación por región,
precisa pero lenta) vs una etapa (YOLO: grilla que regresa cajas en una pasada, tiempo
real). Mask R-CNN añade máscaras; DETR elimina NMS con un transformer.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** `IoU(A,B1) = 16/16 = 1.0` (TP). `IoU(A,B2)`: intersección `[2,4]×[0,4]` =
8; unión `16+16−8 = 24` → `8/24 ≈ 0.33` (**FP**: media caja de desplazamiento ya baja de
0.5). `IoU(A,B3)`: intersección = área(B3) = 4; unión = 16 → `4/16 = 0.25` (FP: caja
contenida pero demasiado pequeña). Solo B1 es TP.

**Ejercicio 2.** Orden por confianza: d1, d2, d3, d4. Se acepta d1; d2 se elimina
(IoU 0.65 > 0.5); d3 se acepta (0.05 y 0.08 ≤ 0.5); d4 se elimina (IoU 0.75 con d3).
Final: `{d1, d3}` → probablemente **2 coches**.

**Ejercicio 3.** (a) Semántica: un mapa de píxeles con etiquetas {oveja, pasto}; las 3
ovejas contiguas forman **una sola región** "oveja". (b) Instancias: 3 máscaras separadas,
una por oveja. Para contar necesitas **instancias**: la semántica no distingue cuántos
objetos componen la región.

**Ejercicio 4.** Implementación y verificación debajo; nota el `max(0, ...)` que evita
intersecciones negativas cuando las cajas no se tocan.


In [ ]:
result = run_lab("perception", seed=62)
assert result["kind"] == "perception"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — IoU verificado con código
def iou(a, b):
    inter_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union else 0.0

A = (0, 0, 4, 4)
for B in [(0, 0, 4, 4), (2, 0, 6, 4), (1, 1, 3, 3)]:
    v = iou(A, B)
    print(B, round(v, 3), "TP" if v > 0.5 else "FP")


In [ ]:
# Ejercicio 2 — NMS verificado con código
def nms(dets, thr=0.5):
    # dets: lista de (nombre, confianza, caja)
    dets = sorted(dets, key=lambda d: -d[1])
    keep = []
    while dets:
        best = dets.pop(0)
        keep.append(best[0])
        dets = [d for d in dets if iou(best[2], d[2]) <= thr]
    return keep

dets = [
    ("d1", 0.95, (0, 0, 10, 10)),
    ("d2", 0.90, (2, 0, 12, 10)),   # IoU alto con d1
    ("d3", 0.85, (20, 0, 30, 10)),  # lejos
    ("d4", 0.40, (21, 0, 31, 10)),  # IoU alto con d3
]
print(nms(dets))  # ['d1', 'd3']


## Reflexión

1. En una escena con dos personas abrazadas, ¿qué falla esperarías de NMS con umbral 0.5 y
   qué cambiarías: el umbral, la métrica o la arquitectura?
2. Tu detector reporta mAP@0.5 = 0.80 en COCO. ¿Qué evidencia adicional necesitas antes de
   afirmar que servirá para contar productos en la estantería de un supermercado?
3. ¿Por qué la confianza de una detección no garantiza que la caja esté bien localizada, y
   cómo lo captura el protocolo de evaluación con IoU?
